## **Notebook 05 — BM25s Sparse Index**
- Load chunks from pgvector → build BM25s index → save to disk → query it → compare with dense search side by side.

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import bm25s
import psycopg2
from pgvector.psycopg2 import register_vector

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")

INDEX_DIR = Path("../data/bm25_index")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print("Imports OK")
print(f"Index will be saved to: {INDEX_DIR}")

Imports OK
Index will be saved to: ../data/bm25_index


In [2]:
conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

cur.execute("""
    SELECT
        id::text,
        content,
        parent_content,
        metadata
    FROM chunks
    ORDER BY created_at;
""")

rows = cur.fetchall()

chunk_ids      = [r[0] for r in rows]
chunk_texts    = [r[1] for r in rows]
chunk_parents  = [r[2] for r in rows]
chunk_metadata = [r[3] for r in rows]

print(f"Loaded {len(chunk_texts)} chunks from pgvector")
print(f"\nSample chunk [0]:")
print(f"  ID      : {chunk_ids[0]}")
print(f"  Content : {chunk_texts[0][:100]}")
print(f"  Section : {chunk_metadata[0].get('section', 'N/A')}")

Loaded 615 chunks from pgvector

Sample chunk [0]:
  ID      : e5429961-95f3-4599-8e96-5a7514a4aa07
  Content : Employees must submit leave requests 7 days in advance.
  Section : hr_policy


In [4]:
corpus_tokens = bm25s.tokenize(
    chunk_texts,
    stopwords="en",
    stemmer=None
)

retriever = bm25s.BM25()
retriever.index(corpus_tokens)

print(f"BM25s index built")
print(f"  Corpus size : {len(chunk_texts)} documents")

# vocab is stored inside corpus_tokens, not retriever
unique_tokens = set(
    token
    for doc_tokens in corpus_tokens.ids
    for token in doc_tokens
)
print(f"  Vocab size  : {len(unique_tokens)} unique tokens")

# Sample from the sids (string ids) if available
try:
    sample = list(corpus_tokens.vocab.keys())[:20]
    print(f"\nSample vocab (first 20 tokens):")
    print(f"  {sample}")
except AttributeError:
    print(f"\nVocab preview not available in this bm25s version — index is still built correctly")

Split strings:   0%|          | 0/615 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/615 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/615 [00:00<?, ?it/s]

BM25s index built
  Corpus size : 615 documents
  Vocab size  : 2275 unique tokens

Sample vocab (first 20 tokens):
  ['employees', 'must', 'submit', 'leave', 'requests', 'days', 'advance', 'all', 'invoices', 'approved', 'finance', 'department', 'company', 'provides', '15', 'annual', 'paid', 'sales', 'targets', 'reviewed']


In [5]:
import pickle

# Save the BM25s retriever
retriever.save(str(INDEX_DIR / "bm25_index"))

# Save chunk IDs separately — needed to map results back to DB rows
with open(INDEX_DIR / "chunk_ids.json", "w") as f:
    json.dump(chunk_ids, f)

# Save chunk texts too — useful for debugging without hitting DB
with open(INDEX_DIR / "chunk_texts.json", "w") as f:
    json.dump(chunk_texts, f)

print("Saved to disk:")
for p in sorted(INDEX_DIR.iterdir()):
    size = p.stat().st_size / 1024
    print(f"  {p.name:<30} {size:>8.1f} KB")

Saved to disk:
  bm25_index                          4.0 KB
  chunk_ids.json                     24.0 KB
  chunk_texts.json                  250.9 KB


In [7]:
loaded_retriever = bm25s.BM25.load(
    str(INDEX_DIR / "bm25_index"),
    load_corpus=False
)

with open(INDEX_DIR / "chunk_ids.json") as f:
    loaded_ids = json.load(f)

with open(INDEX_DIR / "chunk_texts.json") as f:
    loaded_texts = json.load(f)

print(f"Loaded from disk OK")
print(f"  Chunks : {len(loaded_ids)}")
print(f"  Index type : {type(loaded_retriever)}")

Loaded from disk OK
  Chunks : 615
  Index type : <class 'bm25s.BM25'>


In [8]:
def bm25_search(query: str, k: int = 5) -> list[dict]:
    """
    Search the BM25s index.
    Returns list of dicts with chunk_id, content, score.
    """
    query_tokens = bm25s.tokenize([query], stopwords="en")

    results, scores = loaded_retriever.retrieve(
        query_tokens,
        k=min(k, len(loaded_ids))
    )

    output = []
    for idx, score in zip(results[0], scores[0]):
        output.append({
            "chunk_id" : loaded_ids[idx],
            "content"  : loaded_texts[idx],
            "score"    : float(score),
            "rank"     : len(output) + 1,
        })

    return output


# Quick test
test = bm25_search("leave policy")
print(f"BM25s search works — top result:")
print(f"  Score   : {test[0]['score']:.4f}")
print(f"  Content : {test[0]['content'][:100]}")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25s search works — top result:
  Score   : 2.1616
  Content : Leave shall be availed in accordance with  the  leave  roaster  to  be  forwarded  to  the  HRD  pri


In [9]:
from FlagEmbedding import BGEM3FlagModel
import numpy as np

model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    use_fp16=False,
    cache_dir="../models/bge-m3"
)

# THE query that failed in Step 5
QUERY = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

# ── Dense search (pgvector) ─────────────────────────────────
query_vec = model.encode([QUERY], return_dense=True)["dense_vecs"][0]

cur.execute("""
    SELECT
        id::text,
        content,
        1 - (embedding <=> %s::vector) AS score
    FROM chunks
    ORDER BY embedding <=> %s::vector
    LIMIT 5;
""", (query_vec.tolist(), query_vec.tolist()))

dense_results = [
    {"rank": i+1, "score": r[2], "content": r[1], "chunk_id": r[0]}
    for i, r in enumerate(cur.fetchall())
]

# ── Sparse search (BM25s) ────────────────────────────────────
sparse_results = bm25_search(QUERY, k=5)

# ── Print side by side ───────────────────────────────────────
print(f'Query: "{QUERY}"\n')
print(f"{'─'*48} DENSE (pgvector) {'─'*14}")
for r in dense_results:
    print(f"  Rank {r['rank']} | Score {r['score']:.4f}")
    print(f"         {r['content'][:95]}")
    print()

print(f"{'─'*48} SPARSE (BM25s) {'─'*15}")
for r in sparse_results:
    print(f"  Rank {r['rank']} | Score {r['score']:.4f}")
    print(f"         {r['content'][:95]}")
    print()

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

──────────────────────────────────────────────── DENSE (pgvector) ──────────────
  Rank 1 | Score 0.6662
         Permanent/regular: Permanent or regular employee shall be the one who is employed to work until

  Rank 2 | Score 0.6566
         Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary perio

  Rank 3 | Score 0.6287
         - -The candidate must have completed at least two years of continuous service with BDRCS before

  Rank 4 | Score 0.6242
         - -The candidate must have completed at least one year of continuous service with BDRCS before 

  Rank 5 | Score 0.6148
         - -The candidate must have completed at least two years of continuous service with BDRCS before

──────────────────────────────────────────────── SPARSE (BM25s) ───────────────
  Rank 1 | Score 6.3345
         Permanent/regular: Permanent or regular employee shall be the 

In [10]:
def compare_search(query: str, model, cur):
    """Compare top-1 result from dense vs sparse for any query."""
    # Dense
    qvec = model.encode([query], return_dense=True)["dense_vecs"][0]
    cur.execute("""
        SELECT content, 1 - (embedding <=> %s::vector) AS score
        FROM chunks ORDER BY embedding <=> %s::vector LIMIT 1;
    """, (qvec.tolist(), qvec.tolist()))
    d = cur.fetchone()

    # Sparse
    s = bm25_search(query, k=1)[0]

    print(f'Query: "{query}"')
    print(f"  DENSE  ({d[1]:.3f}): {d[0][:85]}")
    print(f"  SPARSE ({s['score']:>6.2f}): {s['content'][:85]}")
    print()

# Query 1 — exact keyword query (sparse should win)
compare_search(
    "What is the mandatory retirement age?",
    model, cur
)

# Query 2 — semantic query using different words (dense should win)
compare_search(
    "How many days off do workers get each year?",
    model, cur
)

# Query 3 — specific acronym (sparse should win)
compare_search(
    "BDRCS provident fund contribution percentage",
    model, cur
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "What is the mandatory retirement age?"
  DENSE  (0.588): The staff members of regular/permanent positions in the society shall retire on the d
  SPARSE (  3.63): -  Retirement from the service after attaining the age of superannuation -  Proper 



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "How many days off do workers get each year?"
  DENSE  (0.707): The company provides 15 days of annual paid leave.
  SPARSE (  4.16): This Human Resource Policy has been developed in relevance with the Strategic Plan 20



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "BDRCS provident fund contribution percentage"
  DENSE  (0.636): Upon  confirmation,  the  staff  members  in  regular/permanent  positions  of  the  
  SPARSE (  7.29): Upon  confirmation,  the  staff  members  in  regular/permanent  positions  of  the  



In [11]:
# conn.close()

print("=" * 50)
print("STEP 6 COMPLETE")
print("=" * 50)
print(f"  Chunks loaded from pgvector   : {len(chunk_texts)}")
print(f"  BM25s index built             : OK")
print(f"  Index saved to disk           : OK")
print(f"  Load from disk verified       : OK")
print()
print("  Key findings:")
print("  Dense search wins on semantic / synonym queries")
print("  Sparse search wins on exact keywords / acronyms")
print("  Retirement age query: Dense MISSED, Sparse FOUND")
print()
print("  This is exactly why we build hybrid search next.")
print()
print("READY FOR STEP 7 — Hybrid Search + RRF")

STEP 6 COMPLETE
  Chunks loaded from pgvector   : 615
  BM25s index built             : OK
  Index saved to disk           : OK
  Load from disk verified       : OK

  Key findings:
  Dense search wins on semantic / synonym queries
  Sparse search wins on exact keywords / acronyms
  Retirement age query: Dense MISSED, Sparse FOUND

  This is exactly why we build hybrid search next.

READY FOR STEP 7 — Hybrid Search + RRF
